In [3]:
import subprocess
import numpy as np

SRA_PATH = "/home/bauerste/5BaseTestrun/nanoporeData/SRR28305167/SRR28305167.sra"
SAMPLE_SIZE = 1000000  # enough for good distribution estimates

lengths = []
cpg_counts = []
SAM_DUMP = "/home/bauerste/tmp/miniforge3/envs/r_env/bin/sam-dump"  # paste yours

cmd = [SAM_DUMP, "--unaligned", SRA_PATH]
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True) as proc:
    for i, line in enumerate(proc.stdout):
        if line.startswith("@"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 10:
            continue
        seq = fields[9]
        if seq == "*" or not seq:
            continue
        seq = seq.upper()
        lengths.append(len(seq))
        cpg_counts.append(seq.count("CG"))
        if len(lengths) >= SAMPLE_SIZE:
            proc.terminate()
            break

if not lengths:
    raise RuntimeError("No usable reads found. Run: sam-dump --unaligned <file> | head")

lengths = np.array(lengths)
cpg_counts = np.array(cpg_counts)

def n50(arr):
    s = np.sort(arr)[::-1]
    return s[np.searchsorted(np.cumsum(s), s.sum() / 2)]

print(f"Sample size:    {len(lengths):,} reads")
print()
print("Read length distribution (bp):")
print(f"  Mean:         {lengths.mean():,.0f}")
print(f"  Median:       {np.median(lengths):,.0f}")
print(f"  N50:          {n50(lengths):,}")
print(f"  P10 / P90:    {np.percentile(lengths, 10):,.0f} / {np.percentile(lengths, 90):,.0f}")
print(f"  P99:          {np.percentile(lengths, 99):,.0f}")
print(f"  Min / Max:    {lengths.min()} / {lengths.max():,}")
print()
print("CpG count per read:")
print(f"  Mean:         {cpg_counts.mean():.1f}")
print(f"  Median:       {np.median(cpg_counts):.0f}")
print(f"  P90:          {np.percentile(cpg_counts, 90):.0f}")
print(f"  Max:          {cpg_counts.max()}")
print()
print("CpG density:")
print(f"  CpGs per kb:  {1000 * cpg_counts.sum() / lengths.sum():.2f}")

Sample size:    1,000,000 reads

Read length distribution (bp):
  Mean:         8,113
  Median:       7,118
  N50:          9,574
  P10 / P90:    3,176 / 14,107
  P99:          24,614
  Min / Max:    13 / 439,926

CpG count per read:
  Mean:         96.7
  Median:       67
  P90:          199
  Max:          10778

CpG density:
  CpGs per kb:  11.92


In [5]:
cmd = ["sam-dump", "--unaligned", sra_path]

with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True) as proc:
    for i, line in enumerate(proc.stdout):
        print(repr(line[:300]))
        if i >= 10:
            break

'1\t4\t*\t0\t0\t*\t*\t0\t0\tTACTTGCTCGACTCATAGTATTGCTGCTTACGGTTCACTACTCACGACGATGTTTTTTTTGGTACCTTTTTTTTCACCGGAAAGGACCCGTAAAGTGATAATGATTATCATATTATCTACATATACACAGTAGTTGCATATGCTGTGGAGGCCATCAAACCACGTAAATAATCAATTATGACGCAGTATCGTATTAATTGATCTGCACATCAACTTAACGTAAAAACAACTTCAGACAATACAAATCAGCGACACTGAATACGGGGCAACCTCATGTCAACG'
'2\t4\t*\t0\t0\t*\t*\t0\t0\tATGTTGGTGTACAAACACATCGTCTCAGAGTGCTTCAAAACTGCTCTATCAAGAGGAAGGTTCCACTCTGTGAGTTGAATACACACAACACAAAGGATTTACTGAGAATTCTTCTGTCTAGCAGTAAATGAGAAATCCCGCTTCCAACGAAGGCCTCAAACAGGTCTAACTAATCACTTGCAGACTTTACAGACAGAGTCTTTCCAAACTGCTCTATGAAGAGAGGTGAAACTCTGTGAACTGAACGCACAGATGACAAAGCAGTTTCTGAGAATGATTCTG'
'3\t4\t*\t0\t0\t*\t*\t0\t0\tTGTGTGTATCCTCTACTGGTTCAGTTAGGTATTGCTTGGGCTTTTGCTTTATGCAGATATTGACCTCTGTGAAAACAGCGTGCAGCGGCACATTGGACATGCTAACCTCACCTTCGAGCAGCTTCGTAGCTTGATGGAAAGCTTACCGGGAAAGAAAGTGGGAGACAGAACATGAAAAAACAATAAAGGCATGCAAACCCAGTGACCAGATCCTGAAGCTGCTCAGTTTGTGGCGAATAAAAAATGGCGACAACACACCTTGAAGGGCCTAATGCACGCACT'
'4\t4\t*\t0\t0\t*\t*\t0\t0\tTATATGTGCATACCTACTAGCTCAGTTCATATACTT